In [16]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence, pad_sequence
import pickle
import datasets
import numpy as np
import pandas as pd
import time
import torch, numpy as np
from torch.utils.data import Dataset

## Pipeline creation and time test

In [17]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [18]:
df = pd.read_parquet("dataset_seq2seq.parquet")

with open("Mapping.pkl", "rb") as f:
    mapping = pickle.load(f)

In [19]:
mapping_inverse = {i: ch for ch, i in mapping.items()}
for j in df["context"][0] :
    print(mapping_inverse[j])

<PART=INTRO>
<lines=1>
<total=1>
Previous :
<START>
Previous :
<START>


## First loader method

In [20]:
dataset = datasets.Dataset.from_pandas(df[["context","text"]])
dataset.set_format(type="torch", columns=["context","text"])

In [21]:
class SeqDataset(Dataset) :
    def __init__(self, data) :
        self.data = data
        self.length = len(data)

    def __len__(self) :
        return self.length

    def __getitem__(self, i) :
        context = self.data["context"][i]
        text = self.data["text"][i]
        return context,text

In [22]:
train_data = SeqDataset(dataset)

In [23]:
def collate_fn(batch) :
    
    contexts, texts = zip(*batch)  # each already a tensor

    length_context = torch.tensor([len(c) for c in contexts], dtype=torch.long)
    length_text = torch.tensor([len(t) for t in texts], dtype=torch.long)

    
    contexts_padded = pad_sequence(contexts, batch_first=True, padding_value=0)
    texts_padded = pad_sequence(texts, batch_first=True, padding_value=0)
    
    texts_X = texts_padded[:,:-1]
    texts_Y = texts_padded[:,1:]
        
    return (contexts_padded, texts_X, texts_Y), (length_context, length_text)

In [24]:
batch_size = 128

train_ld = DataLoader(train_data, batch_size=batch_size, shuffle=False, drop_last=True, collate_fn=collate_fn) 

for j in train_ld :
    break

In [25]:
start = time.time()
for i, batch in enumerate(train_ld):
    if i == 100:   # par ex. sur 100 batchs
        break
end = time.time()
print(f"Temps pour 100 batchs (CPU prep + DataLoader) = {end-start:.2f}s")


Temps pour 100 batchs (CPU prep + DataLoader) = 9.99s


## Second loader method

In [26]:
class MMapSeq2Seq(Dataset):
    def __init__(self, prefix=""):
        self.ctx = np.load(prefix+"context_np.npy","r")
        self.x   = np.load(prefix+"X_np.npy","r")
        self.y   = np.load(prefix+"Y_np.npy","r")

        self.ctx_off = np.load(prefix+"context_offset.npy")
        self.ctx_len = np.load(prefix+"context_length.npy")
        self.x_off   = np.load(prefix+"x_offset.npy")
        self.x_len   = np.load(prefix+"x_length.npy")
        self.y_off   = np.load(prefix+"y_offset.npy")
        self.y_len   = np.load(prefix+"y_length.npy")

    def __len__(self):
        return len(self.ctx_len)

    def __getitem__(self, i):
        # vues memmap -> torch (zéro copie CPU)
        s, L = int(self.ctx_off[i]), int(self.ctx_len[i])
        ctx = torch.from_numpy(self.ctx[s:s+L].astype(np.int32))  # cast léger en vue si besoin

        s, L = int(self.x_off[i]), int(self.x_len[i])
        x = torch.from_numpy(self.x[s:s+L].astype(np.int32))

        s, L = int(self.y_off[i]), int(self.y_len[i])
        y = torch.from_numpy(self.y[s:s+L].astype(np.int32))

        # CrossEntropy/Embedding veulent long (int64) — caste dans collate (mieux) ou ici :
        return ctx, x, y


In [27]:
def collate(batch):
    ctxs, xs, ys = zip(*batch)  # tuples de 1D tensors CPU
    ctxs = [t.to(torch.long, copy=False) for t in ctxs]
    xs   = [t.to(torch.long, copy=False) for t in xs]
    ys   = [t.to(torch.long, copy=False) for t in ys]

    ctx_pad = torch.nn.utils.rnn.pad_sequence(ctxs, batch_first=True, padding_value=PAD_ID)
    x_pad   = torch.nn.utils.rnn.pad_sequence(xs,   batch_first=True, padding_value=PAD_ID)
    y_pad   = torch.nn.utils.rnn.pad_sequence(ys,   batch_first=True, padding_value=PAD_ID)

    ctx_len = torch.tensor([t.numel() for t in ctxs], dtype=torch.int64)  # CPU Long (pack_padded)
    x_len   = torch.tensor([t.numel() for t in xs],   dtype=torch.int64)

    return (ctx_pad, x_pad, y_pad), (ctx_len, x_len)

In [28]:
PAD_ID = 0
ds = MMapSeq2Seq(prefix="files_npy/")  # mets des préfixes train/val si tu split offline

train_ld_1 = DataLoader(
    ds, batch_size=128, shuffle=False, num_workers=0,  # Kaggle: workers=0 pour stabilité
    pin_memory=True, collate_fn=collate, drop_last=True
)


In [30]:
start = time.time()
for i, batch in enumerate(train_ld_1):
    if i == 100:   # par ex. sur 100 batchs
        break
end = time.time()
print(f"Temps pour 100 batchs (CPU prep + DataLoader) = {end-start:.2f}s")

Temps pour 100 batchs (CPU prep + DataLoader) = 0.47s


In [31]:
for i in train_ld_1 :
    break

In [32]:
j

((tensor([[115, 202, 149,  ...,   0,   0,   0],
          [111, 202, 148,  ...,   0,   0,   0],
          [111, 151, 148,  ...,   0,   0,   0],
          ...,
          [113, 191, 129,  ...,   0,   0,   0],
          [112, 202, 122,  ...,   0,   0,   0],
          [112, 151, 122,  ...,   0,   0,   0]]),
  tensor([[216,  87,  43,  ...,   0,   0,   0],
          [216,  75,  79,  ...,   0,   0,   0],
          [216,  75,  71,  ...,   0,   0,   0],
          ...,
          [216,  36,  33,  ...,   0,   0,   0],
          [216,  98,  10,  ...,   0,   0,   0],
          [216,  98,  10,  ...,   0,   0,   0]]),
  tensor([[87, 43, 71,  ...,  0,  0,  0],
          [75, 79, 14,  ...,  0,  0,  0],
          [75, 71, 14,  ...,  0,  0,  0],
          ...,
          [36, 33, 14,  ...,  0,  0,  0],
          [98, 10, 71,  ...,  0,  0,  0],
          [98, 10, 71,  ...,  0,  0,  0]])),
 (tensor([  7,   7,  94,  86, 103, 100, 105, 162, 126,  90,   7, 114, 101,  94,
            7, 114, 101,  94,   7, 107, 

In [33]:
i

((tensor([[115, 202, 149,  ...,   0,   0,   0],
          [111, 202, 148,  ...,   0,   0,   0],
          [111, 151, 148,  ...,   0,   0,   0],
          ...,
          [113, 191, 129,  ...,   0,   0,   0],
          [112, 202, 122,  ...,   0,   0,   0],
          [112, 151, 122,  ...,   0,   0,   0]]),
  tensor([[216,  87,  43,  ...,   0,   0,   0],
          [216,  75,  79,  ...,   0,   0,   0],
          [216,  75,  71,  ...,   0,   0,   0],
          ...,
          [216,  36,  33,  ...,   0,   0,   0],
          [216,  98,  10,  ...,   0,   0,   0],
          [216,  98,  10,  ...,   0,   0,   0]]),
  tensor([[87, 43, 71,  ...,  0,  0,  0],
          [75, 79, 14,  ...,  0,  0,  0],
          [75, 71, 14,  ...,  0,  0,  0],
          ...,
          [36, 33, 14,  ...,  0,  0,  0],
          [98, 10, 71,  ...,  0,  0,  0],
          [98, 10, 71,  ...,  0,  0,  0]])),
 (tensor([  7,   7,  94,  86, 103, 100, 105, 162, 126,  90,   7, 114, 101,  94,
            7, 114, 101,  94,   7, 107, 

As we can see, the new loader method allowes us to gain more than x10 time in loading time on CPU on local and the output are the same. 

Most of the time the CPU is the bottleneck while using Kaggle so I think this will help